In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
from pathlib import Path

notebook_path = Path().absolute()
sys.path.append(str(notebook_path.parent))

In [3]:
import torch
from tqdm import tqdm
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from neural_controllers import NeuralController
from utils import newton_dataset, newton_dataset_new

torch.manual_seed(0)
torch.cuda.manual_seed(0)
np.random.seed(0)

In [4]:
custom_cache_dir = "/scratch/bbjr/skarmakar/huggingface"

model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
model_name='llama_3_8b_it'

language_model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="auto", 
    cache_dir=custom_cache_dir,
)

use_fast_tokenizer = "LlamaForCausalLM" not in language_model.config.architectures
tokenizer = AutoTokenizer.from_pretrained(
    model_id, 
    use_fast=use_fast_tokenizer, 
    padding_side="left", 
    legacy=False,
)

# tokenizer.pad_token_id = 0 if tokenizer.pad_token_id is None else tokenizer.pad_token_id
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [5]:
controller = NeuralController(
    language_model,
    tokenizer,
    rfm_iters=8,
    batch_size=4,
    control_method='rfm'
)

n_components: 5
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 4
M_batch_size         : 2048
n_components         : 5



In [6]:
concept_types = ["Cam", "Isaac"]

data_dir = "../data/prefixed_newton"

dataset = newton_dataset_new(data_dir, controller, concept_types, samples=350)

Train data: 700
Test data: 224
Train data: 700
Test data: 224


In [7]:
rfm_iters = 16
batch_size = 8
n_components = 300
# n_components = 5
energy = 0.99

In [8]:
controllers = {}

for concept_type in tqdm(concept_types):
    
    other_type = [k for k in concept_types if k != concept_type][0]
    
    train_data = dataset[concept_type]['train']
    test_data = dataset[concept_type]['test']
    
    controller = NeuralController(
        language_model,
        tokenizer,
        rfm_iters=rfm_iters,
        batch_size=batch_size,
        control_method='rfm',
        n_components=n_components,
        # energy=energy,
    )
    
    controller.compute_directions(train_data['inputs'], train_data['labels'])
    
    controllers[concept_type] = controller

  0%|          | 0/2 [00:00<?, ?it/s]

n_components: 300
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 16
forward_batch_size   : 8
M_batch_size         : 2048
n_components         : 300

Tuning metric: auc
Getting activations from forward passes


100%|██████████| 70/70 [00:31<00:00,  2.22it/s]


Getting activations from forward passes


100%|██████████| 18/18 [00:07<00:00,  2.53it/s]


...

Time taken to compute eigenvectors: 3.0492584705352783 seconds
train X shape: torch.Size([560, 4096]) train y shape: torch.Size([560, 1]) val X shape: torch.Size([140, 4096]) val y shape: torch.Size([140, 1])
Fitting RFM with ntrain: 560, d: 4096, and nval: 140
Optimal M batch size: 560
Time taken for round 0: 0.005802154541015625 seconds
Optimal M batch size: 560
Time taken for round 1: 0.0068738460540771484 seconds
Optimal M batch size: 560
Time taken for round 2: 0.006962776184082031 seconds
Optimal M batch size: 560
Time taken for round 3: 0.006865978240966797 seconds
Optimal M batch size: 560
Time taken for round 4: 0.006940126419067383 seconds
Optimal M batch size: 560
Time taken for round 5: 0.006916999816894531 seconds
Optimal M batch size: 560
Time taken for round 6: 0.0068895816802978516 seconds
Optimal M batch size: 560
Time taken for round 7: 0.006861686706542969 seconds
Optimal M batch size: 560
Time taken for round 8: 0.006857395172119141 seconds
Optimal M batch size: 560

Time taken to compute eigenvectors: 3.1850359439849854 seconds
train X shape: torch.Size([560, 4096]) train y shape: torch.Size([560, 1]) val X shape: torch.Size([140, 4096]) val y shape: torch.Size([140, 1])
Fitting RFM with ntrain: 560, d: 4096, and nval: 140
Optimal M batch size: 560
Time taken for round 0: 0.005153179168701172 seconds
Optimal M batch size: 560
Time taken for round 1: 0.0068929195404052734 seconds
Optimal M batch size: 560
Time taken for round 2: 0.00690770149230957 seconds
Optimal M batch size: 560
Time taken for round 3: 0.006974458694458008 seconds
Optimal M batch size: 560
Time taken for round 4: 0.006858348846435547 seconds
Optimal M batch size: 560
Time taken for round 5: 0.00688934326171875 seconds
Optimal M batch size: 560
Time taken for round 6: 0.006838321685791016 seconds
Optimal M batch size: 560
Time taken for round 7: 0.006851673126220703 seconds
Optimal M batch size: 560
Time taken for round 8: 0.0068662166595458984 seconds
Optimal M batch size: 560
T

100%|██████████| 31/31 [01:59<00:00,  3.84s/it]

Time taken to compute eigenvectors: 3.2342233657836914 seconds



100%|██████████| 2/2 [05:56<00:00, 178.39s/it]


In [9]:
u = torch.rand((4096, 300))
m = torch.rand((300))

print(u.shape)
print(m.shape)

torch.Size([4096, 300])
torch.Size([300])


In [10]:
l = [1,2,3]
print(u[:,l].shape)
print(m[l].shape)

torch.Size([4096, 3])
torch.Size([3])


In [11]:
path = f"../directions/stable/isaac_cam_{n_components}"

os.makedirs(path, exist_ok=True)


for concept_type in concept_types:
    controller = controllers[concept_type]
    # other_type = [k for k in concept_types if k!=concept_type][0]
    
    controller.save(concept=f'{concept_type}', model_name='llama_3_8b_it', path=path)